# Homework 5 — Deploying Machine Learning Models (2026)

Frozen lead-scoring release. Artifact: `pipeline.bin` (DictVectorizer + LogisticRegression, scikit-learn 1.7.2).

See `FAQ.md` for detailed explanations and gotchas.

## Setup

```bash
uv sync --locked
uv run python smoke_test.py
```

Verify the artifact checksum:

In [ ]:
!sha256sum pipeline.bin

Expected: `1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin`

## Q1 — environment check

In [ ]:
!uv --version

Answer: `uv 0.12.12` (setup check, not graded).

## Q2 — locked dependency

In [ ]:
!grep -i scikit pyproject.toml uv.lock | head

Answer: `scikit-learn==1.7.2`.

## Q3 — load the model

In [ ]:
import pickle
from model import normalize_record

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with open("pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

record = normalize_record(lead)
prob = float(pipeline.predict_proba([record])[0, 1])
print("raw:", prob)
print("rounded 3:", round(prob, 3))

Answer: `0.533`.

## Q4 — serve the model

Start the API in a terminal:

```bash
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

Then send the second lead:

In [ ]:
import json, urllib.request

lead2 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}
req = urllib.request.Request(
    "http://localhost:9696/predict",
    data=json.dumps(lead2).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req) as r:
    print(json.load(r))

Answer: `0.770` (raw `0.769799`).

## Q5 — container configuration

In [ ]:
!grep '^FROM' Dockerfile

Answer: `python:3.11.15-slim-bookworm`.

## Q6 — run the container

In [ ]:
!docker build -t zoomcamp-model:2026-hw5 . 2>&1 | tail -3

In [ ]:
!docker run -d --rm -p 9696:9696 --name hw5 zoomcamp-model:2026-hw5

In [ ]:
!python q6_test.py

In [ ]:
!curl -s http://localhost:9696/health

In [ ]:
!docker stop hw5

Answer: `0.770` (container reproduces Q4; health reports matching checksums).

## Summary of answers

| Q | Answer |
|---|--------|
| Q1 | `0.12.12` (uv version, setup check) |
| Q2 | `1.7.2` |
| Q3 | `0.533` |
| Q4 | `0.770` |
| Q5 | `python:3.11.15-slim-bookworm` |
| Q6 | `0.770` |